In [3]:
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvisa
import qcodes as qc
from qcodes.instrument_drivers.Keithley import Keithley2400

In [4]:
VISA_ADDRESS = "GPIB1::24::INSTR"  # <-- edit to match the address found above
smu = Keithley2400("smu", VISA_ADDRESS)

Connected to: KEITHLEY INSTRUMENTS INC. 2400 (serial:0734512, firmware:C17   Jul 16 1999 15:20:40/A02  /G/F) in 0.35s


In [18]:
# Setting up SMU instrument.

CURRENT_COMPLIANCE = 1e-9 
NPLC = 1
VOLT_RANGE = 20

smu.mode.set('VOLT')
smu.sense.set('CURR')

# Set voltage range.
smu.rangev.set(VOLT_RANGE)
smu.compliancei.set(CURRENT_COMPLIANCE)

# Set integration time.
smu.nplci(NPLC)

print(f"source mode        : {smu.mode.get()}")
print(f"sense function     : {smu.sense.get()}")
print(f"current compliance : {smu.compliancei() * 1e9:.3f} nA")
print(f"abort threshold    : {CURRENT_COMPLIANCE * 1e9:.3f} nA")
print(f"output enabled     : {smu.output.get()}")
print(f"output voltage     : {smu.volt.get()}")

source mode        : VOLT
sense function     : "VOLT:DC","CURR:DC"
current compliance : 1.000 nA
abort threshold    : 1.000 nA
output enabled     : True
output voltage     : -0.0006562138


In [ ]:
v_start = 0
v_stop = -1
v_step = 1e-3
dwell_time = 0.1

num_points = int(np.abs(np.round((v_stop-v_start)/v_step)) + 1)
setpoints = np.linspace(v_start,v_stop,num_points)

input_line = "2MO resistor."

print(f"input line : {input_line}")
print(f"sweep      : {setpoints[0]:.4f} V -> {setpoints[-1]:.4f} V in {v_step:g} V steps")
print(f"points     : {num_points}")
print(f"estimated time  : {num_points * (dwell_time + 0.05 * NPLC):.1f} s")

input line : 2MO resistor.
sweep      : 0.0000 V -> -1.0000 V in 0.001 V steps
points     : 1001
estimated time  : 150.2 s


In [ ]:
def make_setpoints(
    v_start,
    v_stop,
    v_step
):
    num_points = int(np.abs(np.round((v_stop-v_start)/v_step)) + 1)
    setpoints = np.linspace(v_start,v_stop,num_points)
    return setpoints


def run_iv_sweep(
    smu, 
    v_start,
    v_stop,
    v_step,
    dwell_time,
    input_line,
    sweep_back=False,
    abort_current=CURRENT_COMPLIANCE
    ):

    # Ramp to beginning of sweep.
    v_current = smu.volt.get()
    ramp_setpoints = make_setpoints(v_current,v_start,v_step)

    try:
        for v_set in ramp_setpoints:
            smu.volt(float(v_set))
            time.sleep(dwell_time)
            i_meas = smu.curr.get()

            if abs(i_meas) > abort_current:
                aborted = True
                print(
                    f"ERROR: measured current {i_meas * 1e9:.4f} nA exceeded the "
                    f"{abort_current * 1e9:.3f} nA limit at V = {v_set:.4f} V. "
                )
                break
    except KeyboardInterrupt:
        aborted = True